In [ ]:
import json
import os
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve

sys.path.append('..')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
)
from utils.analysis_utils import extract_verdict

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]

# Cache for combined_results
_combined_results_cache = {}


In [ ]:
def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    _combined_results_cache[cache_key] = combined_results
    return combined_results

def prepare_multiple_monitors_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    multiple_monitors_df = load_monitoring_data(base_dir=base_dir)
    num_responses = len(multiple_monitors_df.data.iloc[0].keys())
    for i in range(1, num_responses + 1):
        multiple_monitors_df[f'suspiciousness_{i}_multiple_monitors'] = multiple_monitors_df.data.apply(
            lambda x: extract_verdict(x[f'response_{i}']['content'] if f'response_{i}' in x and isinstance(x[f'response_{i}'], dict) and 'content' in x[f'response_{i}'] else x[f'response_{i}'] if isinstance(x, dict) and f'response_{i}' in x else np.nan, tag_word='verdict')
        )
    multiple_monitors_df.dropna(inplace=True)
    for i in range(1, num_responses + 1):
        multiple_monitors_df[f'max_suspiciousness_{i}_multiple_monitors'] = np.max(multiple_monitors_df[[f'suspiciousness_{j}_multiple_monitors' for j in range(1, i + 1)]], axis=1)
        multiple_monitors_df[f'min_suspiciousness_{i}_multiple_monitors'] = np.min(multiple_monitors_df[[f'suspiciousness_{j}_multiple_monitors' for j in range(1, i + 1)]], axis=1)
    
    return multiple_monitors_df

def merge_and_compute_features_for_multiple_monitors(combined_results: pd.DataFrame, 
                             multiple_monitors_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        multiple_monitors_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features
    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    
    return merged_df


def plot_multiple_monitors_results(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, give_hint: bool, monitors: List[str] = ['claude-3-5-haiku-20241022', 'gpt-4o'], agents: List[str] = ['claude-3-5-sonnet-20241022'], metrics: List[str] = ['final_suspiciousness']):
    """Main execution function."""
    # Configuration
    
    # Load and prepare data
    combined_results = load_combined_results(include_failed_attempts, include_failed_main_task)
    multiple_monitors_df = prepare_multiple_monitors_data(base_dir, include_failed_attempts, include_failed_main_task)
    merged_df = merge_and_compute_features_for_multiple_monitors(combined_results, multiple_monitors_df)
    merged_df = merged_df[merged_df.main_agent.isin(agents)]
    # Print data shapes for verification
    print("Shape of df before merge:", multiple_monitors_df.shape)
    print("Shape of combined_results before merge:", combined_results.shape)
    print("Shape of merged_df after merge:", merged_df.shape)
    
    # Generate plots
    plot_suspiciousness_distributions(merged_df, monitors=monitors, metrics=metrics)
    plot_roc_curves(merged_df, combined_results, give_hint, include_failed_attempts, include_failed_main_task, monitors=monitors, metrics=metrics)
    
    # for monitor in monitors:
    #     plot_markov_analysis(merged_df, main_agent='claude-3-5-sonnet-20241022', monitor_model=monitor, give_hint=False, metrics=metrics)

In [ ]:
if __name__ == "__main__":
    plot_multiple_monitors_results(base_dir='monitor_responses/multiple_monitors_2', include_failed_attempts=False, include_failed_main_task=False, give_hint=True, monitors=['gpt-4o'], metrics=['default_suspiciousness'] + [f'max_suspiciousness_{i}_multiple_monitors' for i in range(1, 10)])
    #plot_read_the_notes_monitoring_results(base_dir='monitor_responses/read_the_running_scratchpad', include_failed_attempts=False, include_failed_main_task=False, give_hint=True, monitors=['gpt-4o'])
    combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=False)
    multiple_monitors_df_shit_prompt = prepare_multiple_monitors_data(base_dir='monitor_responses/multiple_monitors', include_failed_attempts=False, include_failed_main_task=False)
    multiple_monitors_df_chad_prompt = prepare_multiple_monitors_data(base_dir='monitor_responses/multiple_monitors_2', include_failed_attempts=False, include_failed_main_task=False)

    #%%
    merged_df_shit_prompt = merge_and_compute_features_for_multiple_monitors(combined_results, multiple_monitors_df_shit_prompt)
    merged_df_chad_prompt = merge_and_compute_features_for_multiple_monitors(combined_results, multiple_monitors_df_chad_prompt)
    # I WANT TO COMPARE THE PERFORMANCE OF THESE TWO MONITORS
    
    # Inner merge the two dataframes on specified columns
    comparison_df = pd.merge(
        merged_df_shit_prompt,
        merged_df_chad_prompt,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 
            'main_task_evaluation', 'side_task_evaluation', 'main_task_success'],
        how='inner',
        suffixes=('_shit_prompt', '_chad_prompt')
    )
    
    # Display the shape of the merged dataframe
    print(f"Shape of comparison_df after merge: {comparison_df.shape}")
    
    # Display the columns to verify
    print("Columns in the merged dataframe:")
    print(comparison_df.columns.tolist())
    
    plot_roc_curves(comparison_df, combined_results, give_hint=False, include_failed_attempts=False, include_failed_main_task=False, monitors=['gpt-4o'], metrics=[f'suspiciousness_{i}_multiple_monitors{suffix}' for i in range(1, 4) for suffix in ['_shit_prompt', '_chad_prompt']])
    #%%
    merged_df_chad_prompt['std_of_suspiciousness'] = merged_df_chad_prompt.data.apply(lambda x: np.std([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') for y in x.values()]))
    merged_df_shit_prompt['std_of_suspiciousness'] = merged_df_shit_prompt.data.apply(lambda x: np.std([extract_verdict(y["content"] if isinstance(y, dict) and "content" in y else y, tag_word='verdict') for y in x.values()]))
    def plot_std_of_suspiciousness(merged_df: pd.DataFrame, type_of_df):
        # Group by task_name and side_task_evaluation
        # For each group, calculate statistics about the standard deviations
        grouped_stats = merged_df.groupby(['task_name', 'side_task_evaluation'])['std_of_suspiciousness'].agg(['mean', 'count', 'std']).reset_index()
        
        # Calculate 95% confidence interval (CI)
        # For 95% CI, we use 1.96 * standard error of the mean
        grouped_stats['sem'] = grouped_stats['std'] / np.sqrt(grouped_stats['count'])
        grouped_stats['ci_95'] = 1.96 * grouped_stats['sem']
        
        # Create a bar chart with error bars
        plt.figure(figsize=(14, 8))
        
        # Get unique task names and side task evaluations
        task_names = grouped_stats['task_name'].unique()
        side_task_evals = grouped_stats['side_task_evaluation'].unique()
        
        # Set width of bars and positions
        bar_width = 0.35
        x = np.arange(len(task_names))
        
        # Create bars for each side_task_evaluation with error bars
        for i, eval_type in enumerate(side_task_evals):
            data = grouped_stats[grouped_stats['side_task_evaluation'] == eval_type]
            # Ensure data is in the same order as task_names
            ordered_std_values = []
            ordered_errors = []
            
            for task in task_names:
                task_data = data[data['task_name'] == task]
                if not task_data.empty:
                    # This is the standard deviation value itself
                    ordered_std_values.append(task_data['mean'].values[0])
                    # Error bars show 95% confidence interval
                    ordered_errors.append(task_data['ci_95'].values[0])
                else:
                    ordered_std_values.append(0)
                    ordered_errors.append(0)
            
            plt.bar(x + (i - 0.5) * bar_width, ordered_std_values, bar_width, 
                    label=f'Side Task: {eval_type}', 
                    yerr=ordered_errors, 
                    capsize=5,  # Adds caps to the error bars
                    error_kw={'elinewidth': 1.5, 'capthick': 1.5})  # Customizes error bar appearance
        
        # Add labels, title and legend
        plt.xlabel('Task Name', fontsize = 16)
        plt.ylabel('Standard Deviation of Suspiciousness', fontsize = 16)
        plt.title(f'Mean of Standard Deviation of Suspiciousness by Task and Side Task Evaluation\n(Error bars show 95% confidence intervals)\n({type_of_df})', fontsize = 20)
        plt.xticks(x, task_names, rotation=45, ha='right', fontsize = 16)
        plt.legend()
        plt.tight_layout()
        plt.show()
    plot_std_of_suspiciousness(merged_df_chad_prompt, 'Chad Prompt')
    plot_std_of_suspiciousness(merged_df_shit_prompt, 'Shit Prompt')

In [ ]:
# multiple_monitors_df = prepare_multiple_monitors_data(base_dir='monitor_responses/multiple_monitors', include_failed_attempts=False, include_failed_main_task=False)
# #%%
# merged_df = merge_and_compute_features_for_multiple_monitors(combined_results, multiple_monitors_df)
# #%%
# # Calculate differences between suspiciousness scores and plot histograms

# plot_suspiciousness_distributions(merged_df, monitors=['gpt-4o'], metrics=['1_minus_2', '1_minus_3', '2_minus_3', 'max_updated_suspiciousness'])
# #%%
# merged_df